# Moodify: DeBERTa-v3 and RoBERTa vs BERT, whole-song lyrics regression

Trains **DeBERTa-v3-base** and **RoBERTa-base** to predict valence and arousal from lyrics, using the
same setup as the BERT whole-song model:

- the MERGE lyrics, with the same seed-42 80/10/10 split stratified by quadrant (2,044 / 256 / 256 songs);
- songs cut into 160-token windows with a stride of 120 and at most 8 per song;
- each song predicted as the mean over its windows;
- 5 epochs, batch size 8, learning rate 2e-5, linear warm-up over 10% of steps, gradient clipping at 1.0;
- the best epoch picked on validation song MSE.

Both models are then scored on the same 256 test songs, next to the two BERT models:

- **BERT (current):** trained on the first 160 tokens, read over the whole song. This is the live Moodify model.
- **BERT (whole-song trained):** trained with the windows above.

The BERT test predictions are embedded below (titles and numbers only, no lyrics). A check confirms
this notebook's test split is the same 256 songs in the same order.

**Before running:**
1. In your Google Drive, create a folder `moodify` and upload `va_lyrics.csv` and `va_overlap.csv` into it.
   The lyrics stay in your Drive and this runtime only; nothing here prints or saves lyrics.
2. Choose Runtime › Change runtime type › **T4 GPU**.
3. Choose Runtime › Run all. Expect about 15–30 minutes per model on a T4.

Results (reports and per-song predictions, without lyrics) are written to `MyDrive/moodify/colab_results/`.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/moodify"
MODELS = ["microsoft/deberta-v3-base", "roberta-base"]
SAVE_CHECKPOINTS = False   # True writes each model's best weights (~0.5-0.7 GB each) to Drive
EPOCHS, BATCH_SIZE, LR, SEED = 5, 8, 2e-5, 42
MAX_LEN, STRIDE, MAX_WINDOWS = 160, 120, 8

In [ ]:
!pip -q install "transformers>=4.40" sentencepiece protobuf
from google.colab import drive
drive.mount("/content/drive")
import os, torch
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU, then rerun"
print(torch.cuda.get_device_name(0))
OUT_DIR = os.path.join(DRIVE_DIR, "colab_results"); os.makedirs(OUT_DIR, exist_ok=True)
for f in ["va_lyrics.csv", "va_overlap.csv"]:
    assert os.path.exists(os.path.join(DRIVE_DIR, f)), f"upload {f} to {DRIVE_DIR}"

In [ ]:
# Model and windowing code, identical to train_va.py / train_va_windows.py
import io, json, time
import numpy as np
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
from sklearn.model_selection import train_test_split
from torch import nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

AXES = ["valence", "arousal"]
dev = torch.device("cuda")


class TextRegressor(nn.Module):
    """Encoder + dropout + linear + tanh on the first token's final hidden state. (BERT's pooler is
    pretrained, so the BERT model uses it; RoBERTa's pooler is not in its checkpoint and DeBERTa has none.)"""

    def __init__(self, name, dropout_p=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name).float()  # some hub checkpoints are fp16
        self.drop = nn.Dropout(p=dropout_p)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        h = self.bert(input_ids=input_ids, attention_mask=attention_mask)["last_hidden_state"][:, 0]
        return torch.tanh(self.out(self.drop(h)))


def windows(texts, tok, max_len, stride, max_windows):
    body = max_len - 2
    ids_out, owner = [], []
    for k, text in enumerate(texts):
        ids = tok(str(text), add_special_tokens=False, truncation=False, verbose=False)["input_ids"]
        starts = list(range(0, max(len(ids) - body, 0) + 1, stride))
        if starts[-1] + body < len(ids):
            starts.append(len(ids) - body)
        for s in starts[:max_windows]:
            chunk = [tok.cls_token_id] + ids[s:s + body] + [tok.sep_token_id]
            ids_out.append(chunk + [tok.pad_token_id] * (max_len - len(chunk)))
            owner.append(k)
    input_ids = torch.tensor(ids_out, dtype=torch.long)
    return input_ids, (input_ids != tok.pad_token_id).long(), np.array(owner)


def predict_songs(model, w, n_songs, batch_size):
    ids, mask, owner = w
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            out.append(model(input_ids=ids[i:i + batch_size].to(dev),
                             attention_mask=mask[i:i + batch_size].to(dev)).cpu())
    pred = torch.cat(out).numpy()
    sums = np.zeros((n_songs, 2)); np.add.at(sums, owner, pred)
    return sums / np.bincount(owner, minlength=n_songs)[:, None]


def metrics(pred, true):
    res = {}
    for j, name in enumerate(AXES):
        p, t = pred[:, j], true[:, j]; err = p - t
        res[name] = {"mae": float(np.mean(np.abs(err))), "rmse": float(np.sqrt(np.mean(err ** 2))),
                     "pearson": float(np.corrcoef(p, t)[0, 1]) if np.ptp(p) > 0 else None,
                     "r2": float(1 - np.sum(err ** 2) / np.sum((t - t.mean()) ** 2))}
    return res

In [ ]:
# Data and split (same as the local scripts)
df = pd.read_csv(os.path.join(DRIVE_DIR, "va_lyrics.csv"))
ov = pd.read_csv(os.path.join(DRIVE_DIR, "va_overlap.csv"))
assert len(ov) == len(df) and (ov["title"] == df["title"]).all() and (ov["artist"] == df["artist"]).all()
df["in_cls_data"] = ov["in_cls_data"].to_numpy()
df = df.dropna(subset=["lyrics", "valence", "arousal"])
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["quadrant"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["quadrant"])
print(f"split: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test songs")

# BERT test predictions from the Mac runs (no lyrics)
BERT_CSV = """title,artist,cur_v,cur_a,win_v,win_a
All Through the Night,Ella Fitzgerald,0.2761,-0.1831,0.41561,-0.05706
The Way You Look Tonight,Fred Astaire,0.4786,0.0936,0.42361,-0.08639
Spanish Guitar,Toni Braxton,0.5162,-0.0819,0.51983,-0.0488
The Wedding Cake Walk,Fred Astaire,0.5301,0.1735,0.53364,0.20245
I Go to Pieces,Linda Ronstadt,-0.5313,-0.139,-0.48626,-0.27864
Showdown,Barcode,-0.3885,0.2303,-0.4204,0.18945
War,Bruce Springsteen,-0.3896,-0.0213,-0.31414,-0.00768
Rudolph the Red-Nosed Reindeer,Alan Jackson,0.4914,-0.1514,0.5022,-0.12099
Desire Me,Sam Cooke,0.4348,-0.0624,0.45729,0.0141
In Sufferance,Disincarnate,-0.3889,0.1321,-0.39614,0.06715
The Little Vagabond,Greg Brown,0.4339,-0.2863,0.3085,-0.2352
The Way Worn Traveler,Johnny Cash,0.019,0.0036,0.04843,-0.00884
Chords of Fame,Marianne Faithfull,0.2136,-0.0199,0.24459,0.03133
You Take Part in Creating the System [Bonus Track],Discharge,-0.3466,0.1241,-0.33006,0.13672
How Can We Live Like This,Youth Brigade,-0.5491,-0.0948,-0.46475,-0.03088
If It Weren't For Him,Vince Gill,-0.6237,-0.2455,-0.61129,-0.30824
You'd Be So Nice to Come Home To,Art Pepper,0.4533,-0.1886,0.41336,-0.32097
Evil Woman,Champion Jack Dupree,0.3803,0.2647,0.20514,0.33326
War Pigs,Black Sabbath,-0.4328,0.1609,-0.43169,0.16031
Jimmies Dream,Crumbsuckers,-0.4004,0.106,-0.40338,0.111
Crush 'Em,Megadeth,-0.393,0.2364,-0.31551,0.22904
Let’s Go Crazy,Prince,0.3714,0.2403,0.27777,0.11255
Apocalypse Politics,Joan of Arc,0.0986,-0.0917,0.01895,-0.12848
My City of Ruins,Bruce Springsteen,0.4403,0.1646,0.21183,0.14034
You Make Me Believe,Kenny G,0.575,0.0724,0.53633,-0.05081
House on the Lake,Rosanne Cash,-0.2089,-0.2879,-0.21152,-0.3101
Hard Walking Hanna,Jimmy Reed,0.3601,0.1885,0.07594,0.00536
Dead Stop,Negative Approach,-0.4289,0.064,-0.39528,0.02217
Baby Makin' Hips,Fantasia,0.5175,0.3777,0.4626,0.37621
A Satisfied Mind,Bob Dylan,0.2831,-0.0685,0.33854,-0.07206
Sex. Murder. Art.,Slayer,-0.4071,0.1979,-0.38866,0.18025
I Wanna Grow Up To Be a Politician,The Byrds,0.2288,0.0479,0.11301,0.05718
Wish you were here,Pink Floyd,-0.2612,-0.2917,-0.48684,-0.2496
Love on Your Side (Rap Boy Rap),Thompson Twins,0.2828,0.0469,0.39712,0.16373
Gets Me Through,Ozzy Osbourne,0.0448,-0.0676,-0.10169,-0.02281
"Baby, I'm for Real",The Originals,0.3598,0.0478,0.31344,-0.00941
Foolin' Myself,Billie Holiday,-0.2932,0.0274,-0.27381,0.07896
Laura (What's He Got That I Ain't Got),Leon Ashley,-0.1581,-0.126,-0.33285,-0.18452
That's When I See the Blue (In Your Pretty Brown Eyes),Jim Reeves,-0.3188,-0.1479,-0.38178,-0.17825
Straight to Hell,Flotsam and Jetsam,-0.4557,0.109,-0.5075,0.3399
Let's Wait Awhile,Janet Jackson,0.5615,-0.0625,0.54413,-0.17346
Red Blue Jeans and a Pony Tail,Gene Vincent,0.5872,0.265,0.56462,0.21575
One Voice,Barry Manilow,-0.0073,-0.2861,-0.15489,-0.32431
Moody's Mood for Love,Rachelle Ferrell,0.5713,0.1099,0.4528,-0.00639
Sally Sue Brown,Bob Dylan,-0.0105,0.0165,0.12043,0.06462
The Bleeding,Five Finger Deadth Punch,-0.4607,0.0559,-0.42465,-0.05295
We Are Family,Chic,0.6432,0.232,0.62674,0.13295
Out of Mind,The Cure,0.0068,0.1002,-0.10636,0.06505
Storm in a Teacup,Erasure,-0.249,-0.2834,-0.21465,-0.25505
I Wanna Hold You,Al Green,0.4901,0.1609,0.46042,-0.07835
In Amber,Love as Laughter,-0.326,-0.0569,-0.38777,-0.19804
Lately,Stevie Wonder,-0.4764,-0.3022,-0.44522,-0.33728
The Mexican 2002,Marc Anthony,0.6159,0.4154,0.63805,0.47377
Baby a Go-Go,Barbara McNair,0.5763,0.111,0.47564,0.03116
Almost Like Being in Love,101 Strings,0.512,0.0691,0.37923,-0.04511
Angels Among Us,Alabama,0.3987,-0.3904,0.28177,-0.43683
Modulation,Ani DiFranco,-0.2859,-0.0293,-0.39937,-0.07172
Our Love Is Here to Stay,Rod Stewart,0.3914,-0.2061,0.31701,-0.12274
Gentle on My Mind,Aretha Franklin,0.4963,0.0382,0.49397,-0.02573
Im Happy Just to Dance With You,The Beatles,0.6001,0.1509,0.64761,0.25831
"(I Got Spurs That) Jingle, Jangle, Jingle",Kay Kyser,0.5505,0.1816,0.57876,0.12186
Rio [US Edit],Duran Duran,0.6189,0.1571,0.58591,0.04043
Reaching For the Moon,Lizz Wright,0.2232,-0.4,0.35978,-0.29402
Victory,Megadeth,-0.3956,0.1791,-0.37454,0.1568
Sickness,Obituary,-0.4458,0.1478,-0.45127,0.17625
5D (Fifth Dimension),The Byrds,-0.0376,-0.1488,-0.23474,-0.10551
Gloomy Sunday,Billie Holliday,-0.19,-0.3418,-0.04994,-0.3115
Highway Highway,Joe Cocker,-0.0484,-0.312,-0.10199,-0.27445
I Won't Forget You,Jim Reeves,-0.0619,-0.4432,-0.27093,-0.34622
Don't Cha,Busta Rhymes,0.4428,0.3522,0.4143,0.27948
Nobody Does It Better,Carly Simon,0.1677,-0.251,0.01375,-0.28434
The Drugs Don't Work,The Verve,-0.4312,-0.02,-0.41118,0.04664
Zion in a Vision,Garnett Silk,0.4716,-0.0372,0.4014,-0.30063
All I Want for Christmas Is a Real Good Tan,Kenny Chesney,0.5374,-0.0036,0.51647,-0.03345
Spirit in Black,Slayer,-0.3717,0.0265,-0.33536,-0.02347
Voice of the Voiceless,Rage Against the Machine,-0.3397,0.0138,-0.30118,0.04113
Beautiful Horses,The Blood Brothers,-0.4432,0.2059,-0.41258,0.15468
Ten Commandments of Love,Bob Marley,0.4994,-0.1086,0.39537,-0.20472
Good Luck,Basement Jaxx,0.1,0.0253,0.06551,-0.10927
Everybody Loves the Sunshine,Incognito,0.5808,0.0952,0.60665,0.16589
You Can't Hurry Love,Dixie Chicks,0.2431,-0.1552,0.08289,-0.1176
Eat You Alive,Limp Bizkit,0.1595,0.2754,0.2053,0.29485
Good Hearted Woman,Deana Carter,0.5209,-0.0644,0.5436,-0.08618
Save a Bread,The Dominoes,0.2139,-0.3853,0.32123,-0.36078
This Is Not an Exit,Anthrax,-0.3783,0.0068,-0.32269,0.02228
I've Got a Rock 'N' Roll Heart,Eric Clapton,0.5219,0.1895,0.57541,0.20238
Shot Down,Crumbsuckers,-0.3812,0.1499,-0.402,0.1273
Solitude,Tonny Bennett,-0.4563,-0.2765,-0.50987,-0.21532
You Never Miss Your Water,Lightnin' Hopkins,0.2924,-0.1179,0.37719,-0.03098
Night Fever,Bee Gees,0.5692,0.1525,0.54989,0.07614
Set You Free,Allison Moorer,-0.0521,-0.2158,-0.31958,-0.33164
In Germany Before the War,Marianne Faithfull,-0.0609,-0.3691,0.12181,-0.27225
L-O-V-E,Nat King Cole,0.5579,-0.1477,0.52637,-0.08528
Don't Let Me Lose This Dream,Aretha Franklin,0.2367,0.056,0.3103,0.17658
Heaven Help,Lenny Kravitz,0.1478,-0.0263,0.12151,-0.06021
Cursed Realms (Of the Winterdemons),Sunn O))),-0.2728,-0.0739,-0.37595,-0.00626
Were You There?,Sam Cooke,-0.1163,-0.1272,0.02849,-0.01614
Miracle,C�line Dion,0.1978,-0.1948,0.40209,-0.22407
If I Could Only Fly,Merle Haggard,-0.3891,-0.4398,-0.48308,-0.44716
"Run, N*****",The Last Poets,-0.3224,0.1496,-0.37329,0.15934
GM2 1,Gorilla Biscuits,-0.2985,0.0506,-0.4152,0.12708
Last of the Mohicans,Black Child,-0.2957,0.1019,-0.33694,0.11312
Yellow Submarine,Beatles,0.1425,-0.0711,0.08595,-0.07777
Lost,Drown,-0.4015,-0.272,-0.44583,-0.23592
Dreamland,Bob Marley,0.5122,-0.1389,0.49427,-0.31038
Call My Name,Van Morrison,0.2325,-0.1339,-0.016,-0.24183
Skid Row,Lonnie Brooks,0.0745,0.0548,0.22692,0.07958
Shoe Box,Barenaked Ladies,-0.4391,-0.1962,-0.47993,-0.19599
(There's No Place Like) Home for the Holidays,Carpenters,0.4499,-0.2066,0.35508,-0.26899
Mr. Blue Sky,Electric Light Orchestra,0.4237,-0.1463,0.43398,-0.1148
Sycophantic,Fleshgrind,-0.4199,0.1939,-0.42607,0.13741
Midnight Creeper,Elton John,0.0759,0.0193,-0.28227,-0.12097
Roll 'Em Pete,Jimmy Reed,0.3076,-0.0745,0.09729,-0.19362
Don't Explain,Ella Fitzgerald,0.0774,-0.35,0.40962,-0.29924
Fading Dimensions,Darkane,-0.3559,0.1066,-0.34617,0.07407
A Natural Woman,Mariah Carey,0.5591,0.1788,0.60883,0.19928
A Way,Biohazard,-0.2438,-0.2295,-0.11447,-0.19959
Tryin' Times,Roberta Flack,-0.2285,-0.0395,-0.16417,-0.05529
Worldly Whys,Defecation,-0.296,0.044,-0.35788,0.07658
I Got Stung,Elvis,0.0163,0.071,-0.25078,0.0404
Trav'lin' All Alone,Billie Holiday,-0.4035,-0.3225,-0.02147,-0.12779
I Love This Bar,Toby Keith,0.4818,0.2976,0.50113,0.18383
Wormwood,Easy Rider,-0.2522,-0.1426,-0.24411,-0.21914
The Violet and a Rose,Little Jimmy Dickens,0.0985,-0.2506,0.03785,-0.27274
You'll Think of Me,Keith Urban,-0.2387,-0.2111,-0.38325,-0.2353
Wood Cabin,Saint Etienne,-0.3224,-0.1813,-0.17903,0.06693
The Last Mile of the Way,Sam Cooke,0.3585,-0.2436,0.37666,-0.23772
Ready to Die,The Notorious B.I.G.,-0.3283,0.2385,-0.34213,0.25556
Dirt,Revolting Cocks,-0.3862,0.1396,-0.40386,0.15878
O Come All Ye Faithful,Elvis Presley,0.4042,-0.2364,0.34878,-0.25257
Dreams,The Corrs,-0.1732,-0.1615,-0.15196,-0.21546
The Briar and the Rose,Tom Waits,0.1672,-0.3156,0.18308,-0.29282
Stubborn Kind of Fellow,Marvin Gaye,0.5076,0.2767,0.61674,0.2969
Shiny Things,Tom Waits,0.3552,-0.2202,0.02787,-0.12114
London You're a Lady,The Pogues,-0.3137,-0.0291,-0.04733,0.04437
Help Me Understand,Trace Adkins,-0.2987,-0.3483,-0.45999,-0.37085
Jack Rabbit,Elton John,0.0795,-0.1529,-0.00057,-0.27398
I Ain't With Being Broke,Geto Boys,-0.3444,0.0889,-0.34228,0.0375
Inflikted,Cavalera Conspiracy,-0.3424,0.1242,-0.3916,0.09331
Cause of Death,Obituary,-0.4021,0.0792,-0.39165,0.06001
Something You Got,"Harry Connick, Jr.",0.4623,0.061,0.47939,0.13496
More Trife Life,Mobb Deep,-0.3981,0.1117,-0.35352,0.08166
Writer in the Sun,Donovan,-0.2411,-0.282,-0.19978,-0.25713
They All Laughed,Louis Armstrong,0.1674,-0.1167,0.43671,0.00179
Dread and the Fugitive Mind,Megadeth,-0.3471,0.0573,-0.35024,0.08648
Pink Pedal Pushers,Carl Perkins,0.4063,0.2656,0.36091,0.27943
Ferret & Featherbird,Hamada,0.0467,-0.5038,-0.21772,-0.37233
This Love,Bryan Ferry,0.2843,-0.0443,0.15261,-0.08187
Deeper and Deeper [Shep's Fierce Deeper Dub],Madonna,0.3434,0.0179,0.39374,0.33416
Psychotron,Megadeth,-0.3908,0.0448,-0.38949,0.05056
Blackbird,Paul McCartney,0.209,-0.2714,0.20222,-0.27085
The Moon Was Yellow and the Night Was Young,Fred E. Ahlert,0.6047,-0.1384,0.59451,-0.2578
Silver Bells,Vince Gill,0.4699,-0.2356,0.4673,-0.44036
Time Folding Machine,Defecation,-0.2479,0.0272,-0.21305,-0.06007
Could've Been You,Cher,-0.1066,-0.2596,-0.4233,-0.38831
"What'd I Say, Pts. 1 & 2",Ray Charles,0.5031,0.2805,0.50346,0.2543
How Do You Do?,The BooMTown Rats,0.4827,0.0551,0.45496,0.03733
"Whoopie-Ti-Yi-Yo, Get Along Little Dogies",Woody Guthrie,0.3798,-0.077,0.10212,-0.13143
I Can't Stop,Al Green,0.4089,0.1779,0.39003,0.19321
I'm Real,Jennifer Lopez,0.2308,-0.0426,0.08489,0.06526
Stormy Monday Blues,Billy Eckstine,0.0553,-0.1621,0.03037,-0.23368
Life's Railway to Heaven,Patsy Cline,0.3369,-0.1226,0.37142,-0.16632
Discharged Reason,Nuclear Assault,-0.4084,0.0584,-0.35268,-0.06524
The Kill in Me,P.O.S,-0.3453,-0.026,-0.36467,-0.00377
When Two Worlds Collide,Roger Miller,-0.0721,-0.365,-0.2006,-0.30014
Prince of Darkness,Megadeth,-0.363,0.1515,-0.37807,0.13301
Sick and Tired,Black Sabbath,-0.5216,0.0669,-0.49874,0.03544
Dead Bodies Everywhere,Korn,-0.5161,0.1492,-0.47585,0.12089
Brazilian Rhyme,"Earth, Wind & Fire",0.4432,-0.1506,0.36269,-0.18183
Angie,Rolling Stones,-0.0374,-0.1076,-0.30994,-0.27326
Blood of Heroes,Megadeth,-0.3327,0.1134,-0.24513,0.12682
Young at Heart,Frank Sinatra,0.4752,-0.096,0.48684,-0.25965
Rhythm Is Gonna Get You,Gloria Estefan,0.6107,0.3009,0.6188,0.27844
No Rollin' Blues,John Lee Hooker,0.044,-0.1625,-0.05003,-0.2258
My Kind of Girl,Count Basie Orchestra,0.5159,-0.0167,0.56042,-0.20768
Together (Wherever We Go),Sandra Church,0.4175,-0.1808,0.30543,-0.16151
Sabbath Bloody Sabbath,Black Sabbath,-0.4576,0.0602,-0.4819,0.06599
Rock the House [Edit],Del the Funky Homosapien,0.4643,0.2968,0.30735,0.24447
F**k Wit Me Now,Lil Wayne,-0.3524,0.1371,-0.3154,0.11311
Impure Perfection,Darkane,-0.3653,0.0962,-0.36611,0.04389
One Love,The Prodigy,0.2825,-0.303,0.21707,-0.35125
Come to Rest,Blindside,-0.3548,0.0209,-0.38047,-0.03263
The Truth About Men,Tracy Byrd,-0.2735,0.0207,-0.13288,0.08688
Finger Poppin' Time,Hank Ballard & the Midnighters,0.559,0.2271,0.53937,0.29203
Girl From Ipanema,Stan Getz,0.3197,-0.3527,0.17708,-0.38463
Sometimes,Babes in Toyland,-0.421,-0.0101,-0.40605,-0.0185
Mr. Lucky,Revolting Cocks,0.0097,0.2425,0.0116,0.22873
Do you wanna dance,Beach Boys,0.5933,0.3669,0.59423,0.29831
Cut It Dude,The Queers,0.1693,0.2902,0.29474,0.35354
Is There Anybody There,Scorpions,0.2801,-0.2799,0.34324,-0.25258
Give Me Back That Wig (Wig Wearing Woman),Lightnin' Hopkins,0.2458,0.0821,0.38761,0.13772
Dance with my Father,Luthor Vandross,0.0227,-0.2934,-0.04676,-0.26123
Atlantic City,Bruce Springsteen,-0.0972,-0.0485,-0.15884,-0.11164
Turn on the Night,Kiss,0.5376,0.343,0.58315,0.35363
The Way I Am,Merle Haggard,0.1042,-0.1752,-0.10795,-0.05258
Everything Falls Apart,H�sker D�,-0.3701,-0.0445,-0.42694,-0.05957
The Christmas Waltz,Carpenters,0.519,-0.3334,0.53293,-0.34725
Let's Ride,Ja Rule,-0.2169,0.1642,-0.17805,0.09913
Rider in the Rain,Randy Newman,-0.1677,-0.1771,-0.29543,-0.23029
In the Bleak Mid Winter,Sarah McLachlan,0.2669,-0.3099,0.19549,-0.22329
Life Is So Peculiar,Louis Armstrong,0.0556,-0.2331,-0.14947,-0.31464
Our Love,Natalie Cole,0.608,0.1608,0.54174,0.04116
Don't Make Me,Blake Shelton,-0.1966,-0.2209,-0.22926,-0.19826
Door Peep,Sin�ad O'Connor,0.2314,-0.1833,0.04116,-0.10439
Dedicated,Das EFX,-0.2983,0.1566,-0.31665,0.05561
Good Mourning/Black Friday,Megadeth,-0.372,0.187,-0.35413,0.18161
Plastic Man,The Kinks,0.5915,0.1218,0.5171,0.12055
Reaganomics,D.R.I.,-0.3997,0.1185,-0.37322,0.10661
Fear the Grand Inquisitor,Pungent Stench,-0.3899,0.1394,-0.3186,-0.00027
Danke Schoen,Wayne Newton,0.3776,-0.2183,0.5049,0.09696
Lovely Rita,The Beatles,0.4442,-0.0328,0.46595,-0.05043
Baby Face,Lou Reed,0.0761,0.0361,0.0224,0.03876
Foolin' Myself,Buck Clayton,-0.2279,-0.0157,-0.24302,0.00134
Separation of My Soul,Grimfist,-0.4576,0.182,-0.43384,0.05657
You're Gonna Need My Help,Jimmy Reed,-0.2909,-0.0461,-0.38103,-0.11457
Dirge Inferno,Cradle of Filth,-0.4334,0.1525,-0.37908,0.06907
Stoned Is the Way of the Walk,Cypress Hill,0.176,0.2194,-0.0718,0.12291
Already Dead,Beck,-0.4701,-0.0555,-0.48719,-0.11641
Spit to See the Shine,Babes in Toyland,-0.0439,0.1712,-0.14049,0.0112
Fly me to the Moon,Diana Krall,0.5242,-0.2178,0.50612,-0.30216
Last Night Blues,Lightnin' Hopkins,-0.1898,-0.2564,-0.05123,-0.22559
"New York, New York",Frank Sinatra,0.2449,0.0478,0.00098,0.04462
Mortuary Love Affair,Pungent Stench,-0.3119,0.1783,-0.3994,0.23352
Bad Dreams,Joni Mitchell,-0.3406,0.024,-0.38288,-0.06117
For Your Entertainment,Adam Lambert,0.4913,0.2388,0.34055,0.24632
It Won't Be Long (Till I'll Be Leaving),Roy Acuff,0.3867,-0.1693,0.33448,-0.33269
He's Alive,Dolly Parton,-0.0296,-0.2264,-0.07811,-0.21654
Without Love (There Is Nothing),Tom Jones,-0.4034,-0.1816,-0.25678,-0.22711
Amateur Hour,Sparks,0.5721,0.2144,0.56246,0.18442
She's A Woman,Daryle Singletary,0.0076,-0.1458,-0.09874,-0.13974
Build Me Up,Al Green,0.1094,-0.0275,0.35465,0.07856
I Feel Good,James Brown,0.608,0.3194,0.58573,0.30384
You Should Be Dead,50 Cent,-0.342,0.1504,-0.37003,0.18997
Lookin' for a Love,Bobby Womack,0.3153,0.0759,0.22162,0.06777
You Scared,DJ Paul,-0.3617,0.1436,-0.35269,0.12065
Drama,Calvin Richardson,0.0219,0.0603,-0.07366,0.0773
The Man That Got Away,Ella Fitzgerald,-0.2344,-0.2076,-0.15926,-0.09554
Brother,Alice in Chains,-0.345,-0.4186,-0.42453,-0.43712
Outside My Window,Stevie Wonder,0.551,0.2239,0.30904,0.24467
Gangsta Gangsta,N.W.A,-0.3617,0.1945,-0.34055,0.17142
Mountain to Sound,Spoon,0.0743,-0.0443,0.14835,0.06539
Babe,Glassjaw,-0.4545,-0.0253,-0.45885,0.00673
Sunflower,Glen Campbell,0.596,0.1676,0.65394,0.2651
I Wanna Dance with Somebody,Whitney Houston,0.0164,0.0625,0.05711,-0.03754
What Am I Doing Hangin' 'Round,The Monkees,-0.1941,-0.1844,-0.33331,-0.24266
Hate Worldwide,Slayer,-0.4063,0.2096,-0.46642,0.23237
Who Wants the World?,The Stranglers,-0.2015,-0.0419,-0.26038,-0.01181
Don't Be Light,Air,0.2887,-0.0772,-0.01708,-0.24032
Shall We Dance?,Fred Astaire,0.2625,0.0111,0.15056,-0.05103
Thank You,Xzibit,0.0163,-0.0032,-0.01672,-0.01417
Don't Fade Away,Dead Can Dance,0.4029,-0.3364,0.1977,-0.25669
Grand Ole Opry Song,The Nitty Gritty Dirt Band,0.3281,-0.1527,0.2507,-0.2026
Look at Little Sister,Hank Ballard & the Midnighters,0.5777,-0.0073,0.52207,-0.02433
Silence = Death,Within Reach,-0.4682,0.276,-0.47959,0.35496
Sailing to Philadelphia,Mark Knopfler,0.4519,-0.2754,0.40271,-0.35829
Warhead,Venom,-0.3863,0.1787,-0.41192,0.16993
"""
bert = pd.read_csv(io.StringIO(BERT_CSV))
# One title is masked in the embedded data; its artist is still checked.
title_ok = (bert["title"].to_numpy() == test_df["title"].to_numpy()) | bert["title"].str.contains(r"\*{5}").to_numpy()
assert len(bert) == len(test_df) and title_ok.all() \
    and (bert["artist"].to_numpy() == test_df["artist"].to_numpy()).all(), \
    "test split differs from the BERT runs; results would not be comparable"
print("test split matches the BERT runs: same 256 songs, same order")
y_train = train_df[AXES].to_numpy(); y_val = val_df[AXES].to_numpy(); y_test = test_df[AXES].to_numpy()
preds = {"BERT (current: first-160 trained, whole-song read)": bert[["cur_v", "cur_a"]].to_numpy(),
         "BERT (whole-song trained)": bert[["win_v", "win_a"]].to_numpy(),
         "mean baseline": np.tile(y_train.mean(0), (len(y_test), 1))}

In [ ]:
def train_and_score(name):
    t0 = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)
    tok = AutoTokenizer.from_pretrained(name)
    w_train, w_val, w_test = [windows(d["lyrics"].to_numpy(), tok, MAX_LEN, STRIDE, MAX_WINDOWS)
                              for d in (train_df, val_df, test_df)]
    n_win = len(w_train[0])
    print(f"{name}: {n_win} train / {len(w_val[0])} val / {len(w_test[0])} test windows")
    y_win = torch.tensor(y_train[w_train[2]], dtype=torch.float32)
    model = TextRegressor(name).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    steps = (n_win + BATCH_SIZE - 1) // BATCH_SIZE * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1 * steps), num_training_steps=steps)
    loss_fn = nn.MSELoss(); best = float("inf"); history = []
    for epoch in range(EPOCHS):
        model.train(); perm = torch.from_numpy(np.random.permutation(n_win)); losses = []
        for i in range(0, n_win, BATCH_SIZE):
            b = perm[i:i + BATCH_SIZE]
            loss = loss_fn(model(input_ids=w_train[0][b].to(dev), attention_mask=w_train[1][b].to(dev)),
                           y_win[b].to(dev))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); losses.append(loss.item())
        val_mse = float(np.mean((predict_songs(model, w_val, len(val_df), BATCH_SIZE) - y_val) ** 2))
        history.append({"epoch": epoch + 1, "train_window_mse": float(np.mean(losses)), "val_song_mse": val_mse})
        print(f"  epoch {epoch + 1}: train window mse {np.mean(losses):.4f}  val song mse {val_mse:.4f}"
              f"  ({(time.time() - t0) / 60:.0f} min)")
        if val_mse < best:
            best = val_mse; best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    short = name.split("/")[-1]
    if SAVE_CHECKPOINTS:
        torch.save(best_state, os.path.join(OUT_DIR, f"va_{short}_state.bin"))
    all_w = predict_songs(model, w_test, len(test_df), BATCH_SIZE)
    first_w = predict_songs(model, windows(test_df["lyrics"].to_numpy(), tok, MAX_LEN, STRIDE, 1),
                            len(test_df), BATCH_SIZE)
    report = {"base": name, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "max_len": MAX_LEN,
              "stride": STRIDE, "max_windows": MAX_WINDOWS, "n_train_windows": n_win, "history": history,
              "best_val_song_mse": best, "minutes": round((time.time() - t0) / 60, 1),
              "all_windows": metrics(all_w, y_test), "first_window_only": metrics(first_w, y_test)}
    with open(os.path.join(OUT_DIR, f"va_report_{short}.json"), "w") as f:
        json.dump(report, f, indent=2)
    test_df[["title", "artist", "quadrant", "valence", "arousal", "in_cls_data"]] \
        .assign(pred_valence=all_w[:, 0].round(4), pred_arousal=all_w[:, 1].round(4)) \
        .to_csv(os.path.join(OUT_DIR, f"va_test_predictions_{short}.csv"), index=False)
    del model, opt; torch.cuda.empty_cache()
    return all_w, first_w, report

reports = {}
for name in MODELS:
    all_w, first_w, reports[name] = train_and_score(name)
    label = {"microsoft/deberta-v3-base": "DeBERTa-v3-base", "roberta-base": "RoBERTa-base"}.get(name, name)
    preds[f"{label} (whole-song trained)"] = all_w
    preds[f"{label} (first window only)"] = first_w

In [ ]:
# Comparison on the 256 test songs
def table(mask=None):
    rows = []
    for k, p in preds.items():
        m = metrics(p if mask is None else p[mask], y_test if mask is None else y_test[mask])
        rows.append({"model": k, **{f"{ax} {s}": m[ax][s] for ax in AXES for s in ("pearson", "mae", "r2")}})
    return pd.DataFrame(rows).set_index("model").round(3)

print("All 256 test songs")
display(table())
keep = ~test_df["in_cls_data"].to_numpy()
print(f"Excluding the {int((~keep).sum())} test songs also in the 4-class model's training data ({int(keep.sum())} songs)")
display(table(keep))

In [ ]:
# Paired bootstrap: Pearson difference vs the current BERT model, 2,000 resamples of the test songs.
# If the 95% interval includes 0, the difference is within noise on 256 songs.
rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(y_test), size=(2000, len(y_test)))
ref = preds["BERT (current: first-160 trained, whole-song read)"]
def r(p, t): return np.corrcoef(p, t)[0, 1]
rows = []
for k, p in preds.items():
    if "trained)" not in k or k.startswith("BERT (current"):
        continue
    row = {"model": k}
    for j, ax in enumerate(AXES):
        d = np.array([r(p[s, j], y_test[s, j]) - r(ref[s, j], y_test[s, j]) for s in idx])
        row[f"{ax} Δpearson"] = r(p[:, j], y_test[:, j]) - r(ref[:, j], y_test[:, j])
        row[f"{ax} 95% CI"] = f"[{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]"
    rows.append(row)
summary = pd.DataFrame(rows).set_index("model").round(3)
display(summary)
table().to_csv(os.path.join(OUT_DIR, "comparison_all_test_songs.csv"))
summary.to_csv(os.path.join(OUT_DIR, "comparison_bootstrap_vs_current_bert.csv"))
print("saved to", OUT_DIR)